# Experiment 7 — TopK SAE learning rate and factorial sparsity

This standalone notebook contains two sequential experiments using SAELens
`TopKTrainingSAE` with `k=2`:

1. **Experiment 7A:** at fixed `N=256`, train one SAE seed for each candidate learning
   rate and select a rate using Hungarian 90% recovery and MMCS.
2. **Experiment 7B:** only after 7A is complete, fix its selected learning rate and
   train five SAE seeds at `N = 2, 4, 8, ..., 256`.

Every full SAE run receives exactly **125,000,000 generated activations** with a
training batch size of **16,384**. Each generated sample contains exactly two true
primitives, so `k=2` is fixed a priori rather than tuned using the recovery outcomes.

The two true-feature recovery metrics are:

- one-to-one Hungarian recovery at cosine `>= 0.90`;
- Mean Max Cosine Similarity (MMCS).

This notebook does not require any earlier notebook result or toy-model checkpoint.


## Environment setup: local or Google Colab

Run this cell first. Locally, it finds the existing repository checkout and does not
clone, pull, or install packages. In Google Colab it clones the repository into
`/content/SAE` when absent, safely fast-forwards a clean existing clone, and installs
`composed-features/requirements.txt`.

Google Drive is never mounted. Checkpoints, result CSV/JSON files, and PNG plots are
stored inside the checkout under
`composed-features/artifacts/07_topk_sae/`. Colab's `/content` is runtime-local, so
commit, push, or download new artifacts before resetting the runtime.


In [ ]:
from __future__ import annotations

import importlib.util
import os
import subprocess
import sys
from pathlib import Path


try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False

INSTALL_REQUIREMENTS = IN_COLAB
UPDATE_EXISTING_COLAB_REPOSITORY = True
COLAB_REPOSITORY_URL = 'https://github.com/Adefioye/SAE.git'
COLAB_REPOSITORY_ROOT = Path('/content/SAE')


def find_experiment_dir(start: Path) -> Path | None:
    candidates = (start, start / 'composed-features')
    return next(
        (
            path for path in candidates
            if (path / 'sparse_factorial_generator.py').is_file()
        ),
        None,
    )


if IN_COLAB:
    repository_root = COLAB_REPOSITORY_ROOT
    EXPERIMENT_DIR = find_experiment_dir(repository_root)
    if EXPERIMENT_DIR is None:
        if repository_root.exists() and any(repository_root.iterdir()):
            raise FileExistsError(
                f'{repository_root} exists but is not the SAE repository. '
                'Move that directory or choose a different clone path.'
            )
        repository_root.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            [
                'git', 'clone', '--depth', '1',
                COLAB_REPOSITORY_URL, str(repository_root),
            ],
            check=True,
        )
    elif UPDATE_EXISTING_COLAB_REPOSITORY:
        git_directory = repository_root / '.git'
        if git_directory.is_dir():
            repository_status = subprocess.run(
                ['git', '-C', str(repository_root), 'status', '--porcelain'],
                check=True,
                capture_output=True,
                text=True,
            ).stdout.strip()
            if repository_status:
                print('Existing Colab clone has local changes; skipping pull.')
            else:
                subprocess.run(
                    ['git', '-C', str(repository_root), 'pull', '--ff-only'],
                    check=True,
                )
        else:
            print('Existing project is not a Git clone; skipping pull.')
    EXPERIMENT_DIR = find_experiment_dir(repository_root)
else:
    EXPERIMENT_DIR = find_experiment_dir(Path.cwd().resolve())

if EXPERIMENT_DIR is None:
    raise FileNotFoundError(
        'Could not find composed-features/sparse_factorial_generator.py. '
        'Run locally from the repository root/composed-features directory, '
        'or verify the Colab repository URL.'
    )

EXPERIMENT_DIR = EXPERIMENT_DIR.resolve()
REQUIREMENTS_PATH = EXPERIMENT_DIR / 'requirements.txt'
if not REQUIREMENTS_PATH.is_file():
    raise FileNotFoundError(f'Missing requirements file: {REQUIREMENTS_PATH}')

if INSTALL_REQUIREMENTS:
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '--upgrade-strategy',
        'only-if-needed',
        '-r',
        str(REQUIREMENTS_PATH),
    ])

os.chdir(EXPERIMENT_DIR)
print('Runtime:', 'Google Colab' if IN_COLAB else 'local Jupyter')
print('Experiment directory:', EXPERIMENT_DIR)
print('Requirements installed:', INSTALL_REQUIREMENTS)
if IN_COLAB:
    print('Storage: /content runtime-local; Google Drive is not used.')


In [ ]:
import json
import math
import platform
import sys
from collections.abc import Iterator, Sequence
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm


if 'EXPERIMENT_DIR' not in globals():
    working_dir = Path.cwd().resolve()
    if (working_dir / 'sparse_factorial_generator.py').is_file():
        EXPERIMENT_DIR = working_dir
    elif (working_dir / 'composed-features' / 'sparse_factorial_generator.py').is_file():
        EXPERIMENT_DIR = working_dir / 'composed-features'
    else:
        raise FileNotFoundError('Run the Environment setup cell first.')
EXPERIMENT_DIR = Path(EXPERIMENT_DIR).resolve()

if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

import sae_lens
from sae_lens import TopKTrainingSAE, TopKTrainingSAEConfig
from sae_lens.config import LoggingConfig, SAETrainerConfig
from sae_lens.training.sae_trainer import SAETrainer

from sparse_factorial_generator import FactorialConfig, SparseFactorialGenerator


def default_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


DEVICE = default_device()
ARTIFACT_ROOT = EXPERIMENT_DIR / 'artifacts' / '07_topk_sae'
CHECKPOINT_ROOT = ARTIFACT_ROOT / 'sae_checkpoints'
RESULTS_ROOT = ARTIFACT_ROOT / 'results'
for path in (CHECKPOINT_ROOT, RESULTS_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('CUDA device:', torch.cuda.get_device_name(DEVICE))
elif DEVICE.type == 'mps':
    print('Apple Metal acceleration enabled.')
print('SAELens:', sae_lens.__version__)
print('PyTorch:', torch.__version__)
print('Artifact root:', ARTIFACT_ROOT)


## Fixed configuration and execution gates

`RUN_EXPERIMENT_7A` and `RUN_EXPERIMENT_7B` default to `False` to prevent accidental
multi-billion-sample runs. Run 7A first. Once all learning-rate candidates finish, the
notebook writes a validated selection manifest. Experiment 7B refuses to train without
that manifest.

Leave `SELECTED_LEARNING_RATE_OVERRIDE=None` to use the automatic recommendation, or
set it to one of the completed 7A candidates after reviewing both metrics.


In [ ]:
N_VALUES = (2, 4, 8, 16, 32, 64, 128, 256)
TUNING_N = 256
TOPK_K = 2

TRAINING_SAMPLES = 125_000_000
TRAIN_BATCH_SIZE = 16_384

LEARNING_RATE_CANDIDATES = (
    1e-4, 1.5e-4, 2e-4, 3e-4, 4e-4,
    5e-4, 6e-4, 7.5e-4, 1e-3, 1.5e-3,
)

TUNING_SAE_SEED = 0
FIVE_SAE_SEEDS = tuple(range(5))
DICTIONARY_SEED = 0
DATA_SEED = 0

# Shared trainer settings from Experiments 3 and 6.
ADAM_BETA1 = 0.0
ADAM_BETA2 = 0.999

# TopK-specific settings: keep defaults except decoder_init_norm, which follows
# the project-wide Experiment 3/6 initialization.
AUX_LOSS_COEFFICIENT = 1.0
RESCALE_ACTS_BY_DECODER_NORM = True
USE_SPARSE_ACTIVATIONS = False
DECODER_INIT_NORM = 1.0

RUN_EXPERIMENT_7A = False
RUN_EXPERIMENT_7B = False
SELECTED_LEARNING_RATE_OVERRIDE: float | None = None

full_batches, final_batch_size = divmod(TRAINING_SAMPLES, TRAIN_BATCH_SIZE)
EXPECTED_OPTIMIZER_STEPS = full_batches + int(final_batch_size > 0)
assert TOPK_K == 2
assert len(FIVE_SAE_SEEDS) == 5

print('TopK k:', TOPK_K)
print('Tuning N:', TUNING_N)
print('Learning-rate candidates:', LEARNING_RATE_CANDIDATES)
print('Samples per SAE:', f'{TRAINING_SAMPLES:,}')
print('Batch size:', f'{TRAIN_BATCH_SIZE:,}')
print('Full batches:', f'{full_batches:,}')
print('Final batch:', f'{final_batch_size:,}')
print('Optimizer steps per SAE:', f'{EXPECTED_OPTIMIZER_STEPS:,}')


## Deterministic orthogonal-antipodal geometry

For every even `N`, including `N=2`, the two factor sets contain unit-norm antipodal
axis pairs in orthogonal subspaces, followed by a seeded random orthogonal rotation.
Each generated sample contains one `x` primitive and one `y` primitive with equal
positive amplitudes. There is no trained-model or prior-artifact dependency.


In [ ]:
def orthogonal_matrix(dim: int, seed: int) -> torch.Tensor:
    generator = torch.Generator(device='cpu').manual_seed(seed)
    matrix = torch.randn(dim, dim, generator=generator)
    q, r = torch.linalg.qr(matrix)
    signs = torch.where(torch.diag(r) >= 0, 1.0, -1.0)
    return q * signs.unsqueeze(0)


def make_antipodal_dictionary(
    n_per_set: int,
    *,
    seed: int,
    device: torch.device | str,
) -> torch.Tensor:
    if n_per_set <= 0 or n_per_set % 2:
        raise ValueError('Antipodal geometry requires a positive, even N.')

    half = n_per_set // 2
    dictionary = torch.zeros(2 * n_per_set, n_per_set)
    for axis in range(half):
        dictionary[2 * axis, axis] = 1.0
        dictionary[2 * axis + 1, axis] = -1.0

        y_offset = n_per_set
        y_axis = half + axis
        dictionary[y_offset + 2 * axis, y_axis] = 1.0
        dictionary[y_offset + 2 * axis + 1, y_axis] = -1.0

    return (dictionary @ orthogonal_matrix(n_per_set, seed)).to(device)


def verify_antipodal_dictionary(
    dictionary: torch.Tensor, n_per_set: int
) -> None:
    unit = F.normalize(dictionary, dim=1)
    x_group = unit[:n_per_set]
    y_group = unit[n_per_set:]
    for group in (x_group, y_group):
        assert torch.allclose(group[0::2], -group[1::2], atol=2e-4)
    assert float((x_group @ y_group.T).abs().max()) < 2e-4


for n_per_set in N_VALUES:
    verify_antipodal_dictionary(
        make_antipodal_dictionary(
            n_per_set, seed=DICTIONARY_SEED, device=DEVICE
        ),
        n_per_set,
    )
print('Verified all generated orthogonal-antipodal dictionaries.')


## TopK SAE and exact-budget training

The main TopK forward pass keeps the two largest encoder preactivations per sample,
then applies ReLU. There is no L1 penalty. Apart from architecture-specific TopK
settings and the swept learning rate, shared SAE/trainer settings match Experiments 3
and 6.


In [ ]:
@dataclass(frozen=True)
class ExperimentSpec:
    n_per_set: int
    sae_seed: int
    learning_rate: float
    dictionary_seed: int = DICTIONARY_SEED
    data_seed: int = DATA_SEED
    training_samples: int = TRAINING_SAMPLES
    batch_size: int = TRAIN_BATCH_SIZE
    k: int = TOPK_K

    @property
    def activation_dim(self) -> int:
        return self.n_per_set

    @property
    def d_sae(self) -> int:
        return 2 * self.n_per_set

    def checkpoint_slug(self) -> str:
        return (
            f'N-{self.n_per_set:04d}'
            f'__dict-{self.dictionary_seed}'
            f'__data-{self.data_seed}'
            f'__sae-{self.sae_seed}'
            f'__T-{self.training_samples}'
            f'__batch-{self.batch_size}'
            f'__k-{self.k}'
            f'__lr-{slug_value(self.learning_rate)}'
        )


def slug_value(value: float) -> str:
    return format(value, '.12g').replace('-', 'm').replace('.', 'p')


def build_dictionary(spec: ExperimentSpec) -> torch.Tensor:
    return make_antipodal_dictionary(
        spec.n_per_set,
        seed=spec.dictionary_seed,
        device=DEVICE,
    )


def build_generator(spec: ExperimentSpec) -> SparseFactorialGenerator:
    return SparseFactorialGenerator(
        FactorialConfig(
            n_x=spec.n_per_set,
            n_y=spec.n_per_set,
            activation_dim=spec.activation_dim,
            amplitude_correlation=1.0,
            singleton_probability=0.0,
            noise_std=0.0,
            seed=spec.data_seed,
        ),
        device=str(DEVICE),
        true_dictionary=build_dictionary(spec),
        normalize_dictionary=False,
    )


class FixedBatchActivationIterator(Iterator[torch.Tensor]):
    def __init__(
        self, generator: SparseFactorialGenerator, batch_size: int
    ) -> None:
        self.generator = generator
        self.batch_size = batch_size

    def __iter__(self):
        return self

    @torch.no_grad()
    def __next__(self) -> torch.Tensor:
        hidden, _ = self.generator.sample(self.batch_size)
        return hidden.detach()


def make_topk_sae(spec: ExperimentSpec) -> TopKTrainingSAE:
    torch.manual_seed(spec.sae_seed)
    cfg = TopKTrainingSAEConfig(
        d_in=spec.activation_dim,
        d_sae=spec.d_sae,
        device=str(DEVICE),
        dtype='float32',
        apply_b_dec_to_input=False,
        normalize_activations='none',
        k=spec.k,
        use_sparse_activations=USE_SPARSE_ACTIVATIONS,
        aux_loss_coefficient=AUX_LOSS_COEFFICIENT,
        rescale_acts_by_decoder_norm=RESCALE_ACTS_BY_DECODER_NORM,
        decoder_init_norm=DECODER_INIT_NORM,
    )
    return TopKTrainingSAE(cfg).to(DEVICE)


def make_trainer(
    spec: ExperimentSpec,
    sae: TopKTrainingSAE,
    generator: SparseFactorialGenerator,
) -> SAETrainer:
    cfg = SAETrainerConfig(
        total_training_samples=spec.training_samples,
        train_batch_size_samples=spec.batch_size,
        lr=spec.learning_rate,
        lr_end=spec.learning_rate,
        lr_scheduler_name='constant',
        lr_warm_up_steps=0,
        lr_decay_steps=0,
        adam_beta1=ADAM_BETA1,
        adam_beta2=ADAM_BETA2,
        device=str(DEVICE),
        n_checkpoints=0,
        checkpoint_path=None,
        save_final_checkpoint=False,
        logger=LoggingConfig(
            log_to_wandb=False,
            eval_every_n_wandb_logs=2**31 - 1,
        ),
        n_batches_for_norm_estimate=1,
    )
    return SAETrainer(
        cfg=cfg,
        sae=sae,
        data_provider=FixedBatchActivationIterator(generator, spec.batch_size),
    )


def train_exact_sample_budget(
    trainer: SAETrainer,
    generator: SparseFactorialGenerator,
    *,
    description: str,
) -> TopKTrainingSAE:
    pbar = tqdm(
        total=TRAINING_SAMPLES,
        initial=trainer.n_training_samples,
        desc=description,
    )
    while trainer.n_training_samples < TRAINING_SAMPLES:
        remaining = TRAINING_SAMPLES - trainer.n_training_samples
        current_batch_size = min(TRAIN_BATCH_SIZE, remaining)
        hidden, _ = generator.sample(current_batch_size)
        trainer.maybe_reset_sparsity()
        step_output = trainer.step(hidden.detach())
        if not bool(torch.isfinite(step_output.loss).all()):
            pbar.close()
            raise FloatingPointError(
                f'Non-finite loss at optimizer step {trainer.n_training_steps:,}.'
            )
        trainer.n_training_steps += 1
        pbar.update(current_batch_size)

    if trainer.n_training_samples != TRAINING_SAMPLES:
        raise RuntimeError('Trainer did not stop at the exact sample budget.')
    trainer.set_final_sae_metadata()
    pbar.close()
    return trainer.sae


In [ ]:
def checkpoint_spec(spec: ExperimentSpec) -> dict:
    return {
        **asdict(spec),
        'activation_dim': spec.activation_dim,
        'd_sae': spec.d_sae,
        'optimizer_steps': EXPECTED_OPTIMIZER_STEPS,
        'architecture': 'topk',
        'apply_b_dec_to_input': False,
        'normalize_activations': 'none',
        'use_sparse_activations': USE_SPARSE_ACTIVATIONS,
        'aux_loss_coefficient': AUX_LOSS_COEFFICIENT,
        'rescale_acts_by_decoder_norm': RESCALE_ACTS_BY_DECODER_NORM,
        'decoder_init_norm': DECODER_INIT_NORM,
        'lr_scheduler_name': 'constant',
        'lr_warm_up_steps': 0,
        'lr_decay_steps': 0,
        'adam_beta1': ADAM_BETA1,
        'adam_beta2': ADAM_BETA2,
        'sae_lens_version': sae_lens.__version__,
    }


def checkpoint_path(spec: ExperimentSpec) -> Path:
    return CHECKPOINT_ROOT / 'topk_k2_125m_generated_antipodal_v1' / spec.checkpoint_slug()


def checkpoint_matches(path: Path, expected: dict) -> bool:
    required = ('cfg.json', 'sae_weights.safetensors', 'experiment_spec.json')
    if not all((path / filename).is_file() for filename in required):
        return False
    try:
        saved = json.loads((path / 'experiment_spec.json').read_text())
    except (OSError, json.JSONDecodeError):
        return False
    return all(saved.get(key) == value for key, value in expected.items())


def train_or_load_sae(
    spec: ExperimentSpec,
) -> tuple[TopKTrainingSAE, str, int]:
    path = checkpoint_path(spec)
    expected = checkpoint_spec(spec)
    if checkpoint_matches(path, expected):
        sae = TopKTrainingSAE.load_from_disk(
            path, device=str(DEVICE), dtype='float32'
        ).to(DEVICE)
        print('Loaded checkpoint:', path.name)
        return sae, 'loaded_checkpoint', EXPECTED_OPTIMIZER_STEPS

    if path.exists():
        raise ValueError(f'Checkpoint exists but does not match this run: {path}')

    generator = build_generator(spec)
    sae = make_topk_sae(spec)
    trainer = make_trainer(spec, sae, generator)
    sae = train_exact_sample_budget(
        trainer,
        generator,
        description=(
            f'N={spec.n_per_set}, seed={spec.sae_seed}, '
            f'lr={spec.learning_rate:g}'
        ),
    )
    path.mkdir(parents=True, exist_ok=False)
    sae.save_model(path)
    expected['device_used_to_train'] = str(DEVICE)
    (path / 'experiment_spec.json').write_text(
        json.dumps(expected, indent=2) + '\n'
    )
    print('Saved checkpoint:', path.name)
    return sae, 'new_training_run', trainer.n_training_steps


def clear_accelerator_cache() -> None:
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    elif DEVICE.type == 'mps':
        torch.mps.empty_cache()


## True-feature recovery

MMCS measures true-primitive-to-nearest-decoder coverage without requiring unique
matches. Hungarian 90% uses the globally optimal one-to-one assignment and reports the
fraction of matched cosine similarities at least `0.90`.


In [ ]:
@torch.no_grad()
def primitive_recovery_metrics(
    sae: TopKTrainingSAE,
    dictionary: torch.Tensor,
) -> dict[str, float]:
    decoder = F.normalize(sae.W_dec.detach().float(), dim=1)
    primitives = F.normalize(dictionary.detach().float(), dim=1)
    similarity = primitives @ decoder.T

    mmcs = float(similarity.max(dim=1).values.mean())
    primitive_rows, decoder_cols = linear_sum_assignment(
        (-similarity).detach().cpu().numpy()
    )
    assigned = similarity[
        torch.as_tensor(primitive_rows, device=similarity.device),
        torch.as_tensor(decoder_cols, device=similarity.device),
    ]
    return {
        'hungarian_90': float((assigned >= 0.90).float().mean()),
        'mmcs': mmcs,
    }


def atomic_save_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix('.tmp')
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def result_row_exists(frame: pd.DataFrame, column: str, value: float) -> bool:
    if frame.empty:
        return False
    return any(
        math.isclose(float(saved), float(value), rel_tol=1e-12, abs_tol=1e-15)
        for saved in frame[column]
    )


def plot_two_metrics(
    frame: pd.DataFrame,
    *,
    x_column: str,
    title: str,
    output_path: Path,
    log_x: bool,
) -> None:
    ordered = frame.sort_values(x_column)
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    for axis, (metric, ylabel) in zip(
        axes,
        (
            ('hungarian_90', 'Hungarian recovery at cosine >= 0.90'),
            ('mmcs', 'Mean Max Cosine Similarity'),
        ),
        strict=True,
    ):
        axis.plot(ordered[x_column], ordered[metric], marker='o', linewidth=2)
        axis.set(title=title, xlabel=x_column, ylabel=ylabel, ylim=(-0.05, 1.05))
        if log_x and x_column == 'n_per_set':
            axis.set_xscale('log', base=2)
            axis.set_xticks(ordered[x_column])
            axis.set_xticklabels([str(int(value)) for value in ordered[x_column]])
            axis.get_xaxis().set_minor_formatter(plt.NullFormatter())
        elif log_x:
            axis.set_xscale('log')
        axis.grid(alpha=0.25)
    figure.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved plot:', output_path)


# Experiment 7A — One-seed TopK learning-rate sweep at `N=256`

Set `RUN_EXPERIMENT_7A=True` in the configuration cell, then run this cell. Results
are saved after every completed learning rate. The automatic recommendation maximizes
the equal-weight mean of Hungarian 90% and MMCS; ties prefer Hungarian 90%, then MMCS,
then proximity to `3e-4`.


In [ ]:
RESULTS_7A_PATH = RESULTS_ROOT / 'experiment_7a_topk_lr_sweep_125m_seed0.csv'
RANKED_7A_PATH = RESULTS_ROOT / 'experiment_7a_topk_lr_sweep_125m_seed0_ranked.csv'
PLOT_7A_PATH = RESULTS_ROOT / 'experiment_7a_topk_lr_sweep_125m_seed0.png'
SELECTION_7A_PATH = RESULTS_ROOT / 'experiment_7a_selected_learning_rate.json'

RESULT_COLUMNS_7A = [
    'experiment', 'n_per_set', 'sae_seed', 'dictionary_seed', 'data_seed',
    'training_samples', 'train_batch_size', 'optimizer_steps', 'k',
    'learning_rate', 'checkpoint_source', 'hungarian_90', 'mmcs',
]


def load_results_7a() -> pd.DataFrame:
    if not RESULTS_7A_PATH.is_file():
        return pd.DataFrame(columns=RESULT_COLUMNS_7A)
    frame = pd.read_csv(RESULTS_7A_PATH)
    missing = set(RESULT_COLUMNS_7A).difference(frame.columns)
    if missing:
        raise ValueError(f'{RESULTS_7A_PATH} is missing: {sorted(missing)}')
    return frame[RESULT_COLUMNS_7A]


def run_experiment_7a() -> pd.DataFrame:
    results = load_results_7a()
    for index, learning_rate in enumerate(LEARNING_RATE_CANDIDATES, start=1):
        if result_row_exists(results, 'learning_rate', learning_rate):
            print(
                f'[{index}/{len(LEARNING_RATE_CANDIDATES)}] '
                f'Reused lr={learning_rate:g}'
            )
            continue

        spec = ExperimentSpec(
            n_per_set=TUNING_N,
            sae_seed=TUNING_SAE_SEED,
            learning_rate=learning_rate,
        )
        sae, checkpoint_source, optimizer_steps = train_or_load_sae(spec)
        metrics = primitive_recovery_metrics(sae, build_dictionary(spec))
        row = {
            'experiment': '7A',
            'n_per_set': spec.n_per_set,
            'sae_seed': spec.sae_seed,
            'dictionary_seed': spec.dictionary_seed,
            'data_seed': spec.data_seed,
            'training_samples': spec.training_samples,
            'train_batch_size': spec.batch_size,
            'optimizer_steps': optimizer_steps,
            'k': spec.k,
            'learning_rate': spec.learning_rate,
            'checkpoint_source': checkpoint_source,
            **metrics,
        }
        results = pd.concat(
            [results, pd.DataFrame([row])], ignore_index=True
        ).sort_values('learning_rate').reset_index(drop=True)
        atomic_save_csv(results[RESULT_COLUMNS_7A], RESULTS_7A_PATH)
        print(
            f'Saved lr={learning_rate:g}: '
            f'H90={metrics["hungarian_90"]:.4f}, MMCS={metrics["mmcs"]:.6f}'
        )
        del sae
        clear_accelerator_cache()
    return results


results_7a = run_experiment_7a() if RUN_EXPERIMENT_7A else load_results_7a()

if not results_7a.empty:
    display(results_7a)
    plot_two_metrics(
        results_7a,
        x_column='learning_rate',
        title='Experiment 7A — TopK k=2 at N=256, SAE seed 0',
        output_path=PLOT_7A_PATH,
        log_x=True,
    )

complete_7a = all(
    result_row_exists(results_7a, 'learning_rate', value)
    for value in LEARNING_RATE_CANDIDATES
)
if complete_7a:
    ranked_7a = results_7a.copy()
    ranked_7a['selection_score'] = (
        ranked_7a['hungarian_90'] + ranked_7a['mmcs']
    ) / 2
    ranked_7a['distance_from_baseline'] = (
        ranked_7a['learning_rate'] - 3e-4
    ).abs()
    ranked_7a = ranked_7a.sort_values(
        ['selection_score', 'hungarian_90', 'mmcs', 'distance_from_baseline'],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    ranked_7a.to_csv(RANKED_7A_PATH, index=False)
    best = ranked_7a.iloc[0]
    selection = {
        'experiment': '7A',
        'architecture': 'topk',
        'k': TOPK_K,
        'tuning_n': TUNING_N,
        'sae_seed': TUNING_SAE_SEED,
        'training_samples_per_sae': TRAINING_SAMPLES,
        'train_batch_size': TRAIN_BATCH_SIZE,
        'learning_rate_candidates': list(LEARNING_RATE_CANDIDATES),
        'selection_rule': (
            'maximize equal-weight mean of hungarian_90 and mmcs; '
            'tie-break by hungarian_90, mmcs, then proximity to 3e-4'
        ),
        'selected_learning_rate': float(best['learning_rate']),
        'hungarian_90': float(best['hungarian_90']),
        'mmcs': float(best['mmcs']),
        'selection_score': float(best['selection_score']),
    }
    SELECTION_7A_PATH.write_text(json.dumps(selection, indent=2) + '\n')
    print('Selected learning rate:', selection['selected_learning_rate'])
    print('Selection manifest:', SELECTION_7A_PATH)
else:
    print('Experiment 7A is incomplete; 7B remains locked.')


## Resolve and validate the learning rate for Experiment 7B

This cell performs no training. A complete 7A selection manifest must match the
current architecture, `k`, `N`, seed, sample budget, batch size, and candidate grid.
An override must be one of the completed 7A learning rates.


In [ ]:
def resolve_selected_learning_rate() -> float:
    if not SELECTION_7A_PATH.is_file():
        raise RuntimeError(
            'Experiment 7B requires a completed Experiment 7A selection manifest. '
            f'Missing: {SELECTION_7A_PATH}'
        )
    selection = json.loads(SELECTION_7A_PATH.read_text())
    expected = {
        'architecture': 'topk',
        'k': TOPK_K,
        'tuning_n': TUNING_N,
        'sae_seed': TUNING_SAE_SEED,
        'training_samples_per_sae': TRAINING_SAMPLES,
        'train_batch_size': TRAIN_BATCH_SIZE,
        'learning_rate_candidates': list(LEARNING_RATE_CANDIDATES),
    }
    mismatches = {
        key: (selection.get(key), value)
        for key, value in expected.items()
        if selection.get(key) != value
    }
    if mismatches:
        raise ValueError(f'Experiment 7A selection mismatch: {mismatches}')

    selected = (
        float(selection['selected_learning_rate'])
        if SELECTED_LEARNING_RATE_OVERRIDE is None
        else float(SELECTED_LEARNING_RATE_OVERRIDE)
    )
    if not any(
        math.isclose(selected, value, rel_tol=1e-12, abs_tol=1e-15)
        for value in LEARNING_RATE_CANDIDATES
    ):
        raise ValueError(
            'SELECTED_LEARNING_RATE_OVERRIDE must be a completed 7A candidate.'
        )
    if not result_row_exists(results_7a, 'learning_rate', selected):
        raise ValueError(f'No completed 7A result exists for lr={selected:g}.')
    return selected


SELECTED_LEARNING_RATE_FOR_7B = None
if RUN_EXPERIMENT_7B:
    SELECTED_LEARNING_RATE_FOR_7B = resolve_selected_learning_rate()
    print('Validated learning rate for 7B:', SELECTED_LEARNING_RATE_FOR_7B)
elif SELECTION_7A_PATH.is_file():
    try:
        SELECTED_LEARNING_RATE_FOR_7B = resolve_selected_learning_rate()
    except (RuntimeError, ValueError) as error:
        print('Experiment 7B remains locked:', error)
    else:
        print('7B-ready learning rate:', SELECTED_LEARNING_RATE_FOR_7B)
else:
    print('Complete Experiment 7A before running Experiment 7B.')


# Experiment 7B — Five-seed TopK sparsity sweep

After reviewing 7A, set `RUN_EXPERIMENT_7B=True` and rerun the configuration, 7A,
selection-resolution, and 7B cells. The selected learning rate and `k=2` remain fixed
while `N` varies. Per-seed progress is saved after every completed SAE. The graphs show
the five-seed mean only.


In [ ]:
RESULT_COLUMNS_7B = [
    'experiment', 'n_per_set', 'sae_seed', 'dictionary_seed', 'data_seed',
    'training_samples', 'train_batch_size', 'optimizer_steps', 'k',
    'learning_rate', 'checkpoint_source', 'hungarian_90', 'mmcs',
]


def paths_7b(learning_rate: float) -> dict[str, Path]:
    suffix = f'lr-{slug_value(learning_rate)}__k-{TOPK_K}'
    return {
        'per_seed': RESULTS_ROOT / f'experiment_7b_per_seed_125m__{suffix}.csv',
        'aggregate': RESULTS_ROOT / f'experiment_7b_aggregate_125m__{suffix}.csv',
        'plot': RESULTS_ROOT / f'experiment_7b_recovery_by_n_125m__{suffix}.png',
        'config': RESULTS_ROOT / f'experiment_7b_run_config_125m__{suffix}.json',
    }


def load_results_7b(path: Path, learning_rate: float) -> dict[tuple[int, int], dict]:
    if not path.is_file():
        return {}
    frame = pd.read_csv(path)
    missing = set(RESULT_COLUMNS_7B).difference(frame.columns)
    if missing:
        raise ValueError(f'{path} is missing: {sorted(missing)}')
    invariants = {
        'training_samples': TRAINING_SAMPLES,
        'train_batch_size': TRAIN_BATCH_SIZE,
        'k': TOPK_K,
        'learning_rate': learning_rate,
        'dictionary_seed': DICTIONARY_SEED,
        'data_seed': DATA_SEED,
    }
    for column, expected in invariants.items():
        if not all(
            math.isclose(float(value), float(expected), rel_tol=1e-12, abs_tol=1e-15)
            for value in frame[column]
        ):
            raise ValueError(f'{path} has incompatible {column} values.')
    completed = {}
    for row in frame[RESULT_COLUMNS_7B].to_dict(orient='records'):
        key = (int(row['n_per_set']), int(row['sae_seed']))
        if key in completed:
            raise ValueError(f'Duplicate saved result key: {key}')
        completed[key] = row
    return completed


def run_experiment_7b(learning_rate: float, path: Path) -> pd.DataFrame:
    specs = [
        ExperimentSpec(
            n_per_set=n_per_set,
            sae_seed=sae_seed,
            learning_rate=learning_rate,
        )
        for n_per_set in N_VALUES
        for sae_seed in FIVE_SAE_SEEDS
    ]
    expected_keys = [(spec.n_per_set, spec.sae_seed) for spec in specs]
    completed = load_results_7b(path, learning_rate)
    unexpected = set(completed).difference(expected_keys)
    if unexpected:
        raise ValueError(f'Unexpected saved result keys: {sorted(unexpected)}')

    for index, spec in enumerate(specs, start=1):
        key = (spec.n_per_set, spec.sae_seed)
        if key in completed:
            print(
                f'[{index}/{len(specs)}] Reused CSV row: '
                f'N={spec.n_per_set}, seed={spec.sae_seed}'
            )
            continue

        sae, checkpoint_source, optimizer_steps = train_or_load_sae(spec)
        metrics = primitive_recovery_metrics(sae, build_dictionary(spec))
        completed[key] = {
            'experiment': '7B',
            'n_per_set': spec.n_per_set,
            'sae_seed': spec.sae_seed,
            'dictionary_seed': spec.dictionary_seed,
            'data_seed': spec.data_seed,
            'training_samples': spec.training_samples,
            'train_batch_size': spec.batch_size,
            'optimizer_steps': optimizer_steps,
            'k': spec.k,
            'learning_rate': spec.learning_rate,
            'checkpoint_source': checkpoint_source,
            **metrics,
        }
        progress = pd.DataFrame(
            [completed[item] for item in expected_keys if item in completed]
        )[RESULT_COLUMNS_7B].sort_values(
            ['n_per_set', 'sae_seed']
        ).reset_index(drop=True)
        atomic_save_csv(progress, path)
        print(
            f'[{index}/{len(specs)}] Saved N={spec.n_per_set}, '
            f'seed={spec.sae_seed}: H90={metrics["hungarian_90"]:.4f}, '
            f'MMCS={metrics["mmcs"]:.6f}'
        )
        del sae
        clear_accelerator_cache()

    missing_runs = [key for key in expected_keys if key not in completed]
    if missing_runs:
        raise RuntimeError(f'Missing completed runs: {missing_runs}')
    return pd.DataFrame(
        [completed[key] for key in expected_keys]
    )[RESULT_COLUMNS_7B].sort_values(
        ['n_per_set', 'sae_seed']
    ).reset_index(drop=True)


def aggregate_five_seeds(frame: pd.DataFrame) -> pd.DataFrame:
    records = []
    expected_seeds = set(FIVE_SAE_SEEDS)
    for n_per_set, group in frame.groupby('n_per_set', sort=True):
        observed = set(group['sae_seed'].astype(int))
        if observed != expected_seeds:
            raise ValueError(
                f'N={int(n_per_set)} has seeds {sorted(observed)}; '
                f'expected {sorted(expected_seeds)}.'
            )
        records.append({
            'n_per_set': int(n_per_set),
            'n_sae_seeds': len(group),
            'training_samples_per_sae': TRAINING_SAMPLES,
            'train_batch_size': TRAIN_BATCH_SIZE,
            'k': TOPK_K,
            'learning_rate': float(group['learning_rate'].iloc[0]),
            'hungarian_90': float(group['hungarian_90'].mean()),
            'mmcs': float(group['mmcs'].mean()),
        })
    return pd.DataFrame(records)


if RUN_EXPERIMENT_7B and SELECTED_LEARNING_RATE_FOR_7B is None:
    raise RuntimeError('Experiment 7B is locked until Experiment 7A is complete.')

if SELECTED_LEARNING_RATE_FOR_7B is not None:
    output_paths_7b = paths_7b(SELECTED_LEARNING_RATE_FOR_7B)
    run_config_7b = {
        'experiment': '7B',
        'architecture': 'topk',
        'k': TOPK_K,
        'selected_learning_rate': SELECTED_LEARNING_RATE_FOR_7B,
        'n_values': list(N_VALUES),
        'sae_seeds': list(FIVE_SAE_SEEDS),
        'training_samples_per_sae': TRAINING_SAMPLES,
        'train_batch_size': TRAIN_BATCH_SIZE,
        'dictionary_seed': DICTIONARY_SEED,
        'data_seed': DATA_SEED,
        'sae_lens_version': sae_lens.__version__,
        'torch_version': torch.__version__,
        'python_version': platform.python_version(),
    }
    output_paths_7b['config'].write_text(
        json.dumps(run_config_7b, indent=2) + '\n'
    )

    if RUN_EXPERIMENT_7B:
        per_seed_results_7b = run_experiment_7b(
            SELECTED_LEARNING_RATE_FOR_7B,
            output_paths_7b['per_seed'],
        )
    elif output_paths_7b['per_seed'].is_file():
        completed_7b = load_results_7b(
            output_paths_7b['per_seed'], SELECTED_LEARNING_RATE_FOR_7B
        )
        per_seed_results_7b = pd.DataFrame(
            completed_7b.values()
        )[RESULT_COLUMNS_7B].sort_values(
            ['n_per_set', 'sae_seed']
        ).reset_index(drop=True)
    else:
        per_seed_results_7b = None
        print('No 7B results yet. Set RUN_EXPERIMENT_7B=True to train.')

    if per_seed_results_7b is not None:
        aggregate_results_7b = aggregate_five_seeds(per_seed_results_7b)
        aggregate_results_7b.to_csv(output_paths_7b['aggregate'], index=False)
        display(aggregate_results_7b)
        plot_two_metrics(
            aggregate_results_7b,
            x_column='n_per_set',
            title=(
                'Experiment 7B — TopK k=2, five-seed mean, '
                f'lr={SELECTED_LEARNING_RATE_FOR_7B:g}'
            ),
            output_path=output_paths_7b['plot'],
            log_x=True,
        )
        print('Saved per-seed results:', output_paths_7b['per_seed'])
        print('Saved aggregate results:', output_paths_7b['aggregate'])
else:
    print('Experiment 7B is locked until Experiment 7A is complete.')


## Interpretation and artifact layout

- Experiment 7A asks which learning rate best recovers the known primitive dictionary
  for a correctly specified `k=2` TopK SAE at `N=256`.
- Experiment 7B asks whether recovery changes as each named primitive becomes rarer
  while per-sample true L0 remains exactly 2.
- High MMCS with lower Hungarian 90% means primitives have nearby decoder directions
  but do not consistently receive distinct decoder slots.

All result files are stored below
`composed-features/artifacts/07_topk_sae/results/`. Final model checkpoints are below
`composed-features/artifacts/07_topk_sae/sae_checkpoints/`. Neither path uses Google
Drive. The 7A `N=256, seed=0` checkpoint at the selected rate is reused by 7B.
